# 履约漏斗

Olist 的订单状态是最终态，一个 delivered 订单在表里只显示 delivered。所以某环节的转化率要按「到达过该环节及之后」的累计口径算。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
sql = """
SELECT
    COUNT(*) AS total_orders,
    SUM(order_status IN ('approved','invoiced','shipped','delivered')) AS paid,
    SUM(order_status IN ('invoiced','shipped','delivered'))            AS invoiced,
    SUM(order_status IN ('shipped','delivered'))                       AS shipped,
    SUM(order_status = 'delivered')                                    AS delivered,
    SUM(order_status IN ('canceled','unavailable'))                    AS lost
FROM dwd_orders
"""
df = query(sql).iloc[0]
for k, v in df.items():
    print(f"{k:14s} {v:>8,}")

total_orders     99,441
paid             97,901
invoiced         97,899
shipped          97,585
delivered        96,478
lost              1,234


In [3]:
total = df.total_orders
steps = ["paid", "invoiced", "shipped", "delivered"]
prev = total
for s in steps:
    print(f"{s:10s} {df[s]/total*100:6.2f}%  环节流失 {(prev-df[s])/prev*100:5.2f}%")
    prev = df[s]
print(f"{'lost':10s} {df.lost/total*100:6.2f}%")

paid        98.45%  环节流失  1.55%
invoiced    98.45%  环节流失  0.00%
shipped     98.13%  环节流失  0.32%
delivered   97.02%  环节流失  1.13%
lost         1.24%


送达率 97.8%，取消与不可用在 1.2% 左右。后面的流失归因只针对这部分订单。